# Online Machine Learning with River 

In [ ]:
from river import datasets, linear_model, metrics, preprocessing, optim, evaluate, compose, tree, forest, neighbors
from river.datasets import synth
import time
mae, rmse, r2 = metrics.MAE(), metrics.RMSE(), metrics.R2()

In [48]:
# Helper function to inspect features metadata 

x, y = next(iter(datasets.Taxis()))

for k, v in x.items():
    print(f"{k:<20} {type(v).__name__:<10} {v}")
print(f"{'TARGET':<20} {type(y).__name__:<10} {y}")

vendor_id            str        2
pickup_datetime      datetime   2016-01-01 00:00:17
passenger_count      int        5
pickup_longitude     float      -73.98174285888672
pickup_latitude      float      40.71915817260742
dropoff_longitude    float      -73.93882751464845
dropoff_latitude     float      40.82918167114258
store_and_fwd_flag   str        N
TARGET               int        849


## Regression

### Taxis Dataset

In [49]:
taxis = datasets.Taxis()

In [84]:
def date_preprocess_taxis(x):
    m = x['pickup_datetime']
    return {
        'hour': m.hour,
        'weekday': m.weekday(),
        'is_weekend': int(m.weekday() >= 5),
    }


#### Linear Regression

In [ ]:
lin_reg_model = compose.Pipeline(
    ('features', compose.FuncTransformer(date_preprocess_taxis)
     + (compose.Select('store_and_fwd_flag') | preprocessing.OneHotEncoder())),
    ('scale', preprocessing.StandardScaler()),
    ('lin_reg', linear_model.LinearRegression(optimizer=optim.Adam(lr=0.01)))
)

start = time.time()
evaluate.progressive_val_score(taxis, lin_reg_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

MAE: 598.237002 RMSE: 5,199.601938 R2: -0.006295
runtime: 36.65644979476929s


#### Tree Based : Hoeffding Adaptive Tree

In [ ]:
hat_model = compose.Pipeline(
    ('features', compose.FuncTransformer(date_preprocess_taxis)),
    ('hat', tree.HoeffdingAdaptiveTreeRegressor(seed=42))
)

start = time.time()
evaluate.progressive_val_score(taxis, hat_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

MAE: 593.654023 RMSE: 5,883.99554 R2: -0.415912
runtime: 41.17053580284119s


#### Ensemble Based : Adaptive Random Forest

In [ ]:
arf_model = compose.Pipeline(
    ('features', compose.FuncTransformer(date_preprocess_taxis)),
    ('arf', forest.ARFRegressor(seed=42))
)

start = time.time()
evaluate.progressive_val_score(taxis, arf_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

MAE: 667.218336 RMSE: 6,279.759172 R2: -0.437638
runtime: 41.304951906204224s


#### KNN Regressor 

In [ ]:
knn_reg_model = compose.Pipeline(
    ('features', compose.FuncTransformer(date_preprocess_taxis)
     + (compose.Select('store_and_fwd_flag') | preprocessing.OneHotEncoder())),
    ('scale', preprocessing.StandardScaler()),
    ('knn', neighbors.KNNRegressor(n_neighbors=5))
)

start = time.time()
evaluate.progressive_val_score(taxis, knn_reg_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

### Bikes Dataset

In [33]:
bikes = datasets.Bikes()

In [77]:
# Dates preprocessing
def date_preprocess_bikes(x):
    m = x['moment']
    return {
        'hour': m.hour,
        'weekday': m.weekday(),
        'is_weekend': int(m.weekday() >= 5),
    }

#### Linear regression

In [ ]:
lin_reg_model = compose.Pipeline(
    ('features', compose.FuncTransformer(date_preprocess_bikes)
     + (compose.Select('station', 'description') | preprocessing.OneHotEncoder())),
    ('scale', preprocessing.StandardScaler()),
    ('lin_reg', linear_model.LinearRegression(optimizer=optim.Adam(lr=0.01)))
)

start = time.time()
evaluate.progressive_val_score(bikes, lin_reg_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

MAE: 574.111531 RMSE: 5,092.925503 R2: -0.005165
runtime: 8.545868873596191s


#### Tree Based : Hoeffding Adaptive Tree

In [ ]:
hat_model = compose.Pipeline(
    ('features', compose.FuncTransformer(date_preprocess_bikes)),
    ('hat', tree.HoeffdingAdaptiveTreeRegressor(seed=42))
)

start = time.time()
evaluate.progressive_val_score(bikes, hat_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

MAE: 6.128518 RMSE: 8.146469 R2: 0.169699
runtime: 4.705096006393433s


#### Ensemble Based : Adaptive Random Forest

In [79]:
arf_model = compose.Pipeline(
    ('features', compose.FuncTransformer(date_preprocess_bikes)),
    ('arf', forest.ARFRegressor(seed=42))
)

start = time.time()
evaluate.progressive_val_score(bikes, arf_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

MAE: 593.655297 RMSE: 5,920.362584 R2: -0.432404
runtime: 60.99817895889282s


#### KNN Regressor 

In [ ]:
knn_reg_model = compose.Pipeline(
    ('features', compose.FuncTransformer(date_preprocess_bikes)
     + (compose.Select('station', 'description') | preprocessing.OneHotEncoder())),
    ('scale', preprocessing.StandardScaler()),
    ('knn', neighbors.KNNRegressor(n_neighbors=5))
)

start = time.time()
evaluate.progressive_val_score(bikes, knn_reg_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

### Stocks Dataset

In [57]:
stocks = datasets.SP500Stocks()

#### Linear Regression

In [58]:
lin_reg_model = compose.Pipeline(
    ('scale', preprocessing.StandardScaler()),
    ('lin_reg', linear_model.LinearRegression(optimizer=optim.Adam(lr=0.01)))
)

start = time.time()
evaluate.progressive_val_score(stocks, lin_reg_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

MAE: 573.951125 RMSE: 5,092.213239 R2: -0.005157
runtime: 0.03524279594421387s


#### Tree Based : Hoeffding Adaptive Tree

In [60]:
hat_model = compose.Pipeline(
    ('hat', tree.HoeffdingAdaptiveTreeRegressor(seed=42))
)

start = time.time()
evaluate.progressive_val_score(stocks, hat_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

MAE: 0.568273 RMSE: 0.787985 R2: -0.017356
runtime: 0.15862011909484863s


#### Ensemble Based : Adaptive Random Forest

In [80]:
arf_model = compose.Pipeline(
    ('arf', forest.ARFRegressor(seed=42))
)

start = time.time()
evaluate.progressive_val_score(stocks, arf_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

MAE: 593.201374 RMSE: 5,918.096555 R2: -0.432371
runtime: 1.019068956375122s


#### KNN Regressor 

In [82]:
knn_reg_model = compose.Pipeline(
    ('scale', preprocessing.StandardScaler()),
    ('knn', neighbors.KNNRegressor(n_neighbors=5))
)

start = time.time()
evaluate.progressive_val_score(stocks, knn_reg_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

MAE: 0.629661 RMSE: 0.849916 R2: -0.183558
runtime: 3.3058130741119385s


### Synthetic Dataset

In [ ]:
friedman = synth.FriedmanDrift(drift_type='gra', position=(3_000, 7_000), seed=42).take(10_000)

#### Linear Regression

In [ ]:
lin_reg_model = compose.Pipeline(
    ('scale', preprocessing.StandardScaler()),
    ('lin_reg', linear_model.LinearRegression(optimizer=optim.Adam(lr=0.01)))
)

start = time.time()
evaluate.progressive_val_score(friedman, lin_reg_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

#### Tree Based : Hoeffding Adaptive Tree

In [ ]:
hat_model = compose.Pipeline(
    ('hat', tree.HoeffdingAdaptiveTreeRegressor(seed=42))
)

start = time.time()
evaluate.progressive_val_score(friedman, hat_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

#### Ensemble Based : Adaptive Random Forest

In [ ]:
arf_model = compose.Pipeline(
    ('arf', forest.ARFRegressor(seed=42))
)

start = time.time()
evaluate.progressive_val_score(friedman, arf_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

#### KNN Regressor

In [ ]:
knn_reg_model = compose.Pipeline(
    ('scale', preprocessing.StandardScaler()),
    ('knn', neighbors.KNNRegressor(n_neighbors=5))
)

start = time.time()
evaluate.progressive_val_score(friedman, knn_reg_model, mae + rmse + r2, show_time=True)
print(mae, rmse, r2)
print(f"runtime: {time.time() - start}s")

---

## Drift Detection

### Datasets

## Classification

### Datasets

In [ ]:
"""
Real 
- Credit Card
- MaliciousURL
- Phishing
- HTTP : Anomaly detection

Synthetic Agrawal

"""